In [1]:
import os
import pickle
import time
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

C:\Users\Suthishna kumar\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

VECTOR_DIR = "../vector_store"

knowledge_index = faiss.read_index(
    os.path.join(VECTOR_DIR, "knowledge_faiss.index")
)

with open(
    os.path.join(VECTOR_DIR, "knowledge_metadata.pkl"),
    "rb"
) as f:
    knowledge_df = pickle.load(f)

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=None
)

llm.eval()

print("Models and knowledge base loaded.")

Models and knowledge base loaded.


In [3]:
def retrieve_knowledge(query, top_k=3):
    
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )
    
    scores, indices = knowledge_index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )
    
    results = []
    
    for score, idx in zip(scores[0], indices[0]):
        
        if idx < len(knowledge_df):
            
            row = knowledge_df.iloc[idx]
            
            results.append({
                "score": float(score),
                "source": row["source"],
                "text": row["text"]
            })
    
    return results

In [4]:
def create_context(question, top_k=3):
    
    results = retrieve_knowledge(
        question,
        top_k=top_k
    )
    
    context = "\n\n".join(
        [
            f"Source: {r['source']}\n{r['text']}"
            for r in results
        ]
    )
    
    return context, results

In [5]:
def basic_prompt(question, context):
    
    return f"""
Answer the following question.

Context:
{context}

Question:
{question}

Answer:
"""

In [6]:
def role_based_prompt(question, context):
    
    return f"""
You are a professional customer support assistant.

Use the provided business knowledge to answer the customer's question.

Business Knowledge:
{context}

Customer Question:
{question}

Provide a clear and helpful answer.
"""

In [7]:
def grounded_prompt(question, context):
    
    return f"""
You are a customer intelligence and support assistant.

Answer the user's question using ONLY the business knowledge provided below.

Rules:
1. Do not invent information.
2. Do not create policies that are not present in the context.
3. If the answer cannot be found in the context, say that the information is not available.
4. Keep the response concise.
5. Use the provided evidence as the basis for the answer.

BUSINESS KNOWLEDGE:
{context}

USER QUESTION:
{question}

GROUNDED ANSWER:
"""

In [8]:
def customer_intelligence_prompt(question, context):
    
    return f"""
You are an AI Customer Intelligence Assistant.

Your task is to help answer customer-related questions using verified business knowledge.

IMPORTANT RULES:
- Use only the supplied evidence.
- Never invent customer information.
- Never invent company policies.
- Do not make predictions yourself.
- If information is missing, clearly state that it is unavailable.
- Separate facts from explanations.
- Keep the final response easy for a support agent or customer to understand.

VERIFIED BUSINESS KNOWLEDGE:
{context}

QUESTION:
{question}

RESPONSE:
"""

In [9]:
def final_production_prompt(question, context):
    
    return f"""
You are the AI Customer Intelligence Assistant for a customer support system.

You receive verified information from internal systems such as:
- Customer database
- SQL analytics
- Machine learning churn prediction
- SHAP explanations
- Business knowledge base

Your responsibility is to explain the verified information clearly.

STRICT RULES:
1. Use only the information provided in the context.
2. Never invent customer details.
3. Never invent churn probabilities.
4. Never calculate or modify ML predictions.
5. Never invent SHAP feature contributions.
6. Never invent company policies.
7. If required information is missing, say that it is unavailable.
8. Keep answers concise and understandable.
9. For policy questions, rely on the supplied business knowledge.
10. For customer questions, rely on the supplied customer data and model outputs.
11. Clearly distinguish verified facts from general explanations.
12. Do not claim that an action was performed unless the context confirms it.

VERIFIED SYSTEM INFORMATION:
{context}

USER QUESTION:
{question}

FINAL ANSWER:
"""

In [10]:
def generate_from_prompt(prompt, max_new_tokens=120):
    
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )
    
    start_time = time.time()
    
    with torch.no_grad():
        
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    elapsed = time.time() - start_time
    
    generated_tokens = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]
    
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )
    
    return answer.strip(), elapsed

In [11]:
question = "What is the cancellation policy?"

context, retrieved = create_context(
    question,
    top_k=3
)

prompts = {
    "Basic": basic_prompt(question, context),
    "Role Based": role_based_prompt(question, context),
    "Grounded RAG": grounded_prompt(question, context),
    "Customer Intelligence": customer_intelligence_prompt(
        question,
        context
    ),
    "Final Production": final_production_prompt(
        question,
        context
    )
}

In [13]:
import torch
prompt_results = []

for name, prompt in prompts.items():
    
    answer, elapsed = generate_from_prompt(prompt)
    
    prompt_results.append({
        "Prompt": name,
        "Answer": answer,
        "Generation_Time": elapsed
    })
    
    print("=" * 80)
    print("PROMPT:", name)
    print("ANSWER:", answer)
    print("TIME:", round(elapsed, 2), "seconds")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROMPT: Basic
ANSWER: The cancellation policy states that customers may request cancellation of their service. Before canceling, customers should review their current contract terms. Support can help understand available options and any applicable contractual conditions. Cancellation fees or specific contract terms cannot be invented; if they are not present in the knowledge base, it will state that the information is unavailable and recommend checking the customer's contract.
TIME: 62.76 seconds
PROMPT: Role Based
ANSWER: The cancellation policy allows customers to request the cancellation of their service. Before canceling, it is recommended that customers review their current contract terms to ensure they understand the available options and any applicable contractual conditions. Contacting support can help clarify these details.
TIME: 38.76 seconds
PROMPT: Grounded RAG
ANSWER: The cancellation policy allows customers to request the cancellation of their service. Before canceling, c

In [14]:
prompt_results_df = pd.DataFrame(
    prompt_results
)

prompt_results_df

,Prompt,Answer,Generation_Time
0,Basic,The cancellation policy states that customers ...,62.755745
1,Role Based,The cancellation policy allows customers to re...,38.757376
2,Grounded RAG,The cancellation policy allows customers to re...,47.991656
3,Customer Intelligence,The cancellation policy allows customers to re...,42.843705
4,Final Production,Customers may request cancellation of their se...,56.617875


In [15]:
evaluation_questions = [
    "What is the cancellation policy?",
    "What billing options are available?",
    "What contract options are available?",
    "How should a high-risk customer be handled?",
    "What should a customer do when they have a technical issue?"
]

evaluation_rows = []

for question in evaluation_questions:
    
    context, retrieved = create_context(
        question,
        top_k=3
    )
    
    prompt = final_production_prompt(
        question,
        context
    )
    
    answer, elapsed = generate_from_prompt(
        prompt
    )
    
    evaluation_rows.append({
        "Question": question,
        "Answer": answer,
        "Sources": ", ".join(
            [r["source"] for r in retrieved]
        ),
        "Generation_Time": elapsed
    })

prompt_evaluation_df = pd.DataFrame(
    evaluation_rows
)

prompt_evaluation_df

,Question,Answer,Sources,Generation_Time
0,What is the cancellation policy?,Customers may request cancellation of their se...,"cancellation_policy.txt, billing_policy.txt, r...",54.191110
1,What billing options are available?,The billing options available depend on the cu...,"billing_policy.txt, support_faq.txt, cancellat...",81.433789
2,What contract options are available?,The cancellation policy states that customers ...,"cancellation_policy.txt, billing_policy.txt, s...",59.187074
3,How should a high-risk customer be handled?,"For high-risk customers, follow these steps:\n...","retention_policy.txt, support_faq.txt, busines...",90.896788
4,What should a customer do when they have a tec...,"When a customer has a technical issue, they sh...","support_faq.txt, business_rules.txt, billing_p...",27.563305


In [16]:
os.makedirs(
    "processed",
    exist_ok=True
)

prompt_evaluation_df.to_csv(
    "processed/prompt_engineering_results.csv",
    index=False
)

print("Prompt evaluation saved.")

Prompt evaluation saved.


In [17]:
prompt_config = {
    "model": MODEL_NAME,
    "embedding_model": "all-MiniLM-L6-v2",
    "retrieval_top_k": 3,
    "max_new_tokens": 120,
    "prompt_strategy": "Final Production Prompt"
}

with open(
    "../models/prompt_config.pkl",
    "wb"
) as f:
    pickle.dump(
        prompt_config,
        f
    )

print("Prompt configuration saved.")

Prompt configuration saved.
